# W2-D1: Alert Correlation — Assignment

Pipeline: Dedup → Session Window → Topology-Aware Grouping

In [2]:
# Cell 1: Load dataset
import json
import networkx as nx
from collections import defaultdict
from datetime import datetime
import os

os.chdir(r'C:\Users\AdminPC\Downloads\aiops-4\week2\aiops-4-w2day-1')

alerts = []
with open('dataset/alerts_sample.jsonl', encoding='utf-8') as f:
    for line in f:
        alerts.append(json.loads(line.strip()))

with open('dataset/services.json', encoding='utf-8') as f:
    svc_data = json.load(f)

print(f'Loaded {len(alerts)} alerts')
print(f'Services in graph: {[s["name"] for s in svc_data["services"]]}')
print()
print('--- Alert sample ---')
for a in alerts[:5]:
    print(f"  {a['id']} | {a['ts']} | {a['service']:20s} | {a['metric']:35s} | {a['severity']}")

Loaded 20 alerts
Services in graph: ['edge-lb', 'auth-svc', 'checkout-svc', 'payment-svc', 'cart-svc', 'catalog-svc', 'recommender-svc', 'inventory-svc', 'notification-svc', 'search-svc']

--- Alert sample ---
  a-0001 | 2026-06-12T09:42:01Z | payment-svc          | db_connection_pool_used_ratio       | warn
  a-0002 | 2026-06-12T09:42:18Z | payment-svc          | db_connection_pool_used_ratio       | crit
  a-0003 | 2026-06-12T09:42:22Z | payment-svc          | latency_p99_ms                      | crit
  a-0004 | 2026-06-12T09:42:30Z | payment-svc          | error_rate                          | warn
  a-0005 | 2026-06-12T09:42:45Z | checkout-svc         | latency_p99_ms                      | warn


In [3]:
# Cell 2: Correlation functions

SEV_RANK = {'warn': 1, 'crit': 2}

def fingerprint(alert):
    """Vân tay alert: service + metric + severity. Không include ts/value vì chúng thay đổi mỗi lần fire."""
    return f"{alert['service']}|{alert['metric']}|{alert['severity']}"

def session_groups(alerts, gap_sec=120):
    """
    Session window: group alert liên tiếp nếu khoảng cách thời gian <= gap_sec.
    Session tự kết thúc khi im lặng > gap_sec giây.
    Chọn gap_sec=120 vì toàn bộ incident span chỉ ~6 phút, burst liên tục.
    """
    if not alerts:
        return []
    sorted_alerts = sorted(alerts, key=lambda a: a['ts'])
    groups = [[sorted_alerts[0]]]
    for alert in sorted_alerts[1:]:
        ts      = datetime.fromisoformat(alert['ts'].replace('Z', '+00:00'))
        last_ts = datetime.fromisoformat(groups[-1][-1]['ts'].replace('Z', '+00:00'))
        if (ts - last_ts).total_seconds() <= gap_sec:
            groups[-1].append(alert)
        else:
            groups.append([alert])
    return groups

def build_graph(svc_data):
    """Build directed service graph từ services.json."""
    g = nx.DiGraph()
    for svc in svc_data['services']:
        g.add_node(svc['name'])
    for store in svc_data['stores']:
        g.add_node(store['name'])
    for edge in svc_data['edges']:
        g.add_edge(edge['from'], edge['to'], type=edge['type'])
    return g

def topology_group(alerts, graph, max_hop=2):
    """
    Union-Find grouping: 2 service cùng cluster nếu shortest_path <= max_hop.
    max_hop=2 giữ được cascade trực tiếp (payment→checkout→edge) mà không
    kéo các service không liên quan ở xa > 2 hop.
    """
    if not alerts:
        return []
    undirected = graph.to_undirected()
    by_service = defaultdict(list)
    for a in alerts:
        by_service[a['service']].append(a)
    services = list(by_service.keys())
    parent = {s: s for s in services}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        parent[find(x)] = find(y)

    for i, s1 in enumerate(services):
        for s2 in services[i+1:]:
            if s1 not in undirected or s2 not in undirected:
                continue
            try:
                dist = nx.shortest_path_length(undirected, s1, s2)
                if dist <= max_hop:
                    union(s1, s2)
            except nx.NetworkXNoPath:
                continue

    groups_dict = defaultdict(list)
    for s in services:
        groups_dict[find(s)].extend(by_service[s])
    return list(groups_dict.values())

def max_sev(group):
    return max(group, key=lambda a: SEV_RANK[a['severity']])['severity']

def correlate(alerts, graph, gap_sec=120, max_hop=2):
    """Main pipeline: session_groups → topology_group → emit clusters."""
    sessions = session_groups(alerts, gap_sec=gap_sec)
    all_clusters = []
    for si, session in enumerate(sessions):
        topo_groups = topology_group(session, graph, max_hop=max_hop)
        for gi, group in enumerate(topo_groups):
            fps = sorted(set(fingerprint(a) for a in group))
            all_clusters.append({
                'cluster_id':   f'c-{si:03d}-{gi:03d}',
                'alert_count':  len(group),
                'services':     sorted(set(a['service'] for a in group)),
                'alert_ids':    sorted(a['id'] for a in group),
                'time_range':   [min(a['ts'] for a in group), max(a['ts'] for a in group)],
                'max_severity': max_sev(group),
                'fingerprints': fps,
            })
    return all_clusters

print('Functions defined: fingerprint, session_groups, build_graph, topology_group, correlate')

Functions defined: fingerprint, session_groups, build_graph, topology_group, correlate


In [5]:
# Cell 3: Run pipeline + write output
import os

graph    = build_graph(svc_data)
clusters = correlate(alerts, graph, gap_sec=120, max_hop=2)

ratio = round(1 - len(clusters) / len(alerts), 4)

print(f'Input alerts   : {len(alerts)}')
print(f'Output clusters: {len(clusters)}')
print(f'Reduction ratio: {ratio}  (need >= 0.5 )' if ratio >= 0.5 else f'Reduction ratio: {ratio}  ❌')
print()
for c in clusters:
    print(f"[{c['cluster_id']}] {c['alert_count']:2d} alerts | sev={c['max_severity']}")
    print(f"   services : {c['services']}")
    print(f"   time     : {c['time_range'][0]} → {c['time_range'][1]}")
    print(f"   alert_ids: {c['alert_ids']}")
    print()

os.makedirs('results', exist_ok=True)
output = {
    'input_alerts':    len(alerts),
    'output_clusters': len(clusters),
    'reduction_ratio': ratio,
    'clusters':        clusters,
}
with open('results/cluster_summary.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print(' results/cluster_summary.json written')

Input alerts   : 20
Output clusters: 1
Reduction ratio: 0.95  (need >= 0.5 )

[c-000-000] 20 alerts | sev=crit
   services : ['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc', 'recommender-svc', 'search-svc']
   time     : 2026-06-12T09:42:01Z → 2026-06-12T09:48:30Z
   alert_ids: ['a-0001', 'a-0002', 'a-0003', 'a-0004', 'a-0005', 'a-0006', 'a-0007', 'a-0008', 'a-0009', 'a-0010', 'a-0011', 'a-0012', 'a-0013', 'a-0014', 'a-0015', 'a-0016', 'a-0017', 'a-0018', 'a-0019', 'a-0020']

 results/cluster_summary.json written
